言：第四篇，给循环装上"挂钩"
上一篇，咱们给 Agent 装了三道闸门：硬拒绝表、规则匹配、人工审批。模型再想跑 rm -rf，先过门禁；被拦下了，拒绝信息回流成它的输入，它会自己改道。看起来齐活了。

但那一篇结尾，我留了一句话没展开：权限检查那个函数，是硬编码在循环里的。check_permission 就焊死在 agent_loop 的工具执行前面，焊死了，就动不了了。

现在问题来了。我想在每次工具执行后记一行日志？想统计这个会话拦了多少危险操作？想让它每次改完文件自动做点收尾动作？按现在的写法，答案只有一个：打开 agent_loop，往里塞代码。

塞一次开一次胸。塞三次，那颗我们从第一篇起就小心翼翼没动过的心脏，就被各种零碎功能糊满了。

这一篇就干一件事：给循环装挂钩。在固定的几个位置留出扩展插口，日志、权限、自动收尾、会话统计，全部挂在外面——循环本体一行不多，功能照长。

读完你会拿到三样东西：

二十行挂钩系统
的完整实现：一张注册表、两个函数、四个事件，往任何位置注入逻辑都不再碰循环
一个工程界一百岁高龄的架构模式：扩展不侵入核心——Web 中间件、git hooks、生命周期钩子，全是这副骨架的不同皮
本篇真正的高潮：Stop hook 强制续命。模型说"我做完了"，挂钩说"你没有"，它只能爬起来继续干——以及这个玩法一不留神就会造出一个永远下不了班的 Agent
门槛不变：会 Python 基础语法、有上一篇跑通的那份带门禁的代码。直接开始。

PART 01：循环发胖了——每加一个功能，就开一次胸
先把现状摆出来看清楚。上一篇装完门禁后，主循环长这样：

In [ ]:
for block in tool_calls:
    print(f"> {block.name}")

    if not check_permission(block):          # 权限检查，焊死在循环里
        results.append({"type": "tool_result",
                        "tool_use_id": block.id,
                        "content": "Permission denied."})
        continue

    handler = TOOL_HANDLERS.get(block.name)
    output = handler(**block.input) if handler else f"Unknown: {block.name}"
    results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})

就一个功能（权限），看起来还算清爽。现在三个新需求砸过来，每一个都真实有用：

审计日志
：每次工具调用前记一行日志——出了事故回头翻账，哪个时间点跑了什么命令，一目了然
自动暂存
：每次 write_file / edit_file 之后自动 git add——Agent 改了什么，暂存区里全程留痕，跑飞了随时能 diff 回来
会话统计
：会话结束时数一数这次一共调了多少次工具、拦了多少次——没有这个，你连自己的 Agent 乖不乖都不知道
按现在的写法怎么实现？打开 agent_loop，塞：

In [ ]:
for block in tool_calls:
    log_to_file(block)                       # 需求1：加一行
    if not check_permission(block):          # 原有：权限
        ...
    output = handler(**block.input)
    auto_git_add(block)                      # 需求2：再加一行
    ...
# 循环退出前
print_session_stats(messages)                # 需求3：再塞一处

三个需求，三处改动，还算能忍。第四个需求呢？第八个呢？

你想扩展的是 Agent 的行为，动的却是它的心脏。每开一次胸就多一分改坏主循环的风险——而主循环是整个系统里唯一"动一行都可能全盘崩掉"的地方。更糟的是，塞着塞着，这个循环很快就没法读了：调度、安全、日志、统计糊成一团，谁也不敢再碰。

这个问题一点都不新鲜，工程界一百年前就在解它，解法你一定见过：

Web 框架的中间件
：Django、Express 都不让你改请求处理主流程，而是在"请求进、响应出"两个位置挂中间件，鉴权、日志、限流全挂外面
git hooks
：git 不让你改它提交代码的流程，但在提交前、提交后留了 .git/hooks 插口，pre-commit 挂上去就能拦坏提交
浏览器的 addEventListener
：页面渲染引擎不认识你的代码，但它在"点击、滚动、输入"这些固定事件上留了监听插口
同一副骨架：核心流程固定不动，在固定位置留插口，扩展逻辑挂上去。插件不进核心，核心不认识插件——两边只通过"插口"这一个薄薄的约定耦合。

给 Agent 的循环装同一副骨架，就是这一篇全部的活。

